# Baseline: Wav2Vec 2.0
## Vietnamese-English Code-Switching ASR
Model: `nguyenvulebinh/wav2vec2-base-vietnamese-250h`

> Vietnamese Wav2Vec2 fine-tuned on 250h Vietnamese speech. CTC-based — no language model.

In [ ]:
!pip install -q \
    transformers==4.40.0 \
    evaluate==0.4.1 \
    jiwer==3.0.3 \
    librosa==0.10.1 \
    soundfile==0.12.1

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import torch
import librosa
from tqdm.notebook import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import evaluate
from jiwer import wer

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_ID      = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
AUDIO_DIR     = "./test/audio"
TRANSCRIPT    = "./test/transcript.json"
OUTPUT_CSV    = "./results_wav2vec2.csv"
SAMPLING_RATE = 16000
BATCH_SIZE    = 8

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────
def load_audio(path, target_sr=SAMPLING_RATE):
    audio, _ = librosa.load(path, sr=target_sr, mono=True)
    return audio[:30 * target_sr] if len(audio) > 30 * target_sr else audio

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def compute_ser(refs, hyps):
    return sum(r.strip() != h.strip() for r, h in zip(refs, hyps)) / len(refs) * 100

def compute_all_metrics(refs, hyps):
    wer_m = evaluate.load("wer")
    cer_m = evaluate.load("cer")
    refs_n = [normalize_text(r) for r in refs]
    hyps_n = [normalize_text(h) for h in hyps]
    return {
        "wer_raw":  round(wer_m.compute(references=refs,   predictions=hyps)   * 100, 2),
        "cer_raw":  round(cer_m.compute(references=refs,   predictions=hyps)   * 100, 2),
        "ser_raw":  round(compute_ser(refs, hyps), 2),
        "wer_norm": round(wer_m.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "cer_norm": round(cer_m.compute(references=refs_n, predictions=hyps_n) * 100, 2),
        "ser_norm": round(compute_ser(refs_n, hyps_n), 2),
    }

def load_test_samples(audio_dir, transcript_path):
    with open(transcript_path, encoding="utf-8") as f:
        transcripts = json.load(f)
    samples, missing = [], 0
    for fname, meta in transcripts.items():
        path = os.path.join(audio_dir, fname)
        if os.path.exists(path):
            samples.append({"filename": fname, "audio_path": path, "text": meta["text"]})
        else:
            missing += 1
    print(f"Loaded {len(samples)} samples (skipped {missing} missing)")
    return samples

print("Helpers defined.")

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
print(f"Loading {MODEL_ID}...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print("Model ready.")

In [ ]:
# ── Load test data ─────────────────────────────────────────────────────────
test_samples = load_test_samples(AUDIO_DIR, TRANSCRIPT)

In [ ]:
# ── Inference ──────────────────────────────────────────────────────────────
filenames, hypotheses, references = [], [], []

for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc="Inference"):
    batch  = test_samples[i : i + BATCH_SIZE]
    audios = [load_audio(s["audio_path"]) for s in batch]

    inputs = processor(
        audios,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
        padding=True,
    )
    input_values = inputs.input_values.to(device).to(next(model.parameters()).dtype)
    attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None

    with torch.no_grad():
        logits = model(
            input_values,
            attention_mask=attention_mask,
        ).logits

    pred_ids = torch.argmax(logits, dim=-1)
    preds = processor.batch_decode(pred_ids)

    hypotheses.extend(preds)
    filenames.extend([s["filename"] for s in batch])
    references.extend([s["text"] for s in batch])

print(f"Done — {len(hypotheses)} samples")

In [ ]:
# ── Metrics & save ─────────────────────────────────────────────────────────
metrics = compute_all_metrics(references, hypotheses)
print(f"Before norm → WER: {metrics['wer_raw']}% | CER: {metrics['cer_raw']}% | SER: {metrics['ser_raw']}%")
print(f"After  norm → WER: {metrics['wer_norm']}% | CER: {metrics['cer_norm']}% | SER: {metrics['ser_norm']}%")

df = pd.DataFrame({"filename": filenames, "reference": references, "hypothesis": hypotheses})
df["wer_per_sentence"] = df.apply(
    lambda r: wer(normalize_text(r["reference"]), normalize_text(r["hypothesis"])) * 100, axis=1
)
df.sort_values("wer_per_sentence", ascending=False).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")